## Imports to run llm

In [2]:
from groq_client import call_llm
from utils import extract_json, save_output, run_case

## Zero Shot Prompting

In [2]:
from prompts import (
    zero_shot_vendor_risk_prompt,
    zero_shot_exec_memo_prompt
)

In [3]:
vendor_result = run_case(
    case_name="zero_shot_vendor_risk",
    prompt_messages=zero_shot_vendor_risk_prompt,
    output_filename="zero_shot_vendor_risk.json"
)


Running: zero_shot_vendor_risk
Saved -> outputs\zero_shot_vendor_risk.json


In [4]:
memo_result = run_case(
    case_name="zero_shot_executive_memo",
    prompt_messages=zero_shot_exec_memo_prompt,
    output_filename="zero_shot_exec_memo.json"
)


Running: zero_shot_executive_memo
Saved -> outputs\zero_shot_exec_memo.json


## Few Shot Prompting

In [1]:
from prompts import (
    few_shot_ticket_classification_prompt,
    few_shot_leave_api_prompt
)

In [3]:
ticket_result = run_case(
    case_name="few_shot_ticket_classification",
    prompt_messages=few_shot_ticket_classification_prompt,
    output_filename="few_shot_ticket_classification.json"
)


Running: few_shot_ticket_classification
Saved -> outputs\few_shot_ticket_classification.json


In [4]:
leave_result = run_case(
    case_name="few_shot_leave_api",
    prompt_messages=few_shot_leave_api_prompt,
    output_filename="few_shot_leave_api.json"
)


Running: few_shot_leave_api
JSON parsing failed: Extra data: line 13 column 1 (char 230)
Saved -> outputs\few_shot_leave_api.json


## Chain of Thoughts

In [1]:
from prompts import (
    cot_roi_decision_prompt,
    cot_ml_root_cause_prompt
)

In [3]:
roi_result = run_case(
    case_name="cot_roi_decision",
    prompt_messages=cot_roi_decision_prompt,
    output_filename="cot_roi_decision.json"
)


Running: cot_roi_decision
Saved -> outputs\cot_roi_decision.json


In [4]:
ml_result = run_case(
    case_name="cot_ml_root_cause",
    prompt_messages=cot_ml_root_cause_prompt,
    output_filename="cot_ml_root_cause.json"
)


Running: cot_ml_root_cause
Saved -> outputs\cot_ml_root_cause.json


## LLM as Judge

In [1]:
from prompts import llm_judge_support_prompt, llm_judge_code_prompt

In [4]:
support_judge_result = run_case(
    case_name="llm_judge_support",
    prompt_messages=llm_judge_support_prompt,
    output_filename="llm_judge_support.json"
)



Running: llm_judge_support
Saved -> outputs\llm_judge_support.json


In [5]:
code_judge_result = run_case(
    case_name="llm_judge_code",
    prompt_messages=llm_judge_code_prompt,
    output_filename="llm_judge_code.json"
)


Running: llm_judge_code
Saved -> outputs\llm_judge_code.json


## Self Consistency

In [ ]:
from collections import Counter
import json


def run_self_consistency(case_name, prompt, output_file, n_runs=5):

    all_outputs = []
    extracted_values = []


    for i in range(n_runs):

        response = call_llm(
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  
        )

        parsed = extract_json(response)

        all_outputs.append({
            "run": i + 1,
            "raw": response,
            "parsed": parsed
        })

        if parsed:
            if "reimbursable_amount" in parsed:
                extracted_values.append(parsed["reimbursable_amount"])
            if "risk_level" in parsed:
                extracted_values.append(parsed["risk_level"])


    if all(isinstance(v, (int, float)) for v in extracted_values):
        final_value = sorted(extracted_values)[len(extracted_values)//2]

        consistency = dict(Counter(extracted_values))

        final_decision = "median_vote"

    else:
        consistency = dict(Counter(extracted_values))
        final_value = max(consistency, key=consistency.get)
        final_decision = "majority_vote"


    result = {
        "runs": all_outputs,
        "risk_level_votes": consistency if "risk_level" in extracted_values else {},
        "final_reimbursable_amount": final_value if isinstance(final_value, (int, float)) else None,
        "final_risk_level": final_value if isinstance(final_value, str) else None,
        "consistency_count": consistency,
        "final_decision": final_decision,
        "reasoning_summary": "Aggregated multiple independent model runs using self-consistency voting."
    }

    save_output(output_file, result)

    return result

In [3]:
from prompts import (
    self_consistency_meal_prompt,
    self_consistency_security_prompt
)

In [4]:
meal_result = run_self_consistency(
    case_name="self_consistency_meal",
    prompt=self_consistency_meal_prompt,
    output_file="self_consistency_meal.json",
    n_runs=5
)

Saved -> outputs\self_consistency_meal.json


In [5]:
security_result = run_self_consistency(
    case_name="self_consistency_security",
    prompt=self_consistency_security_prompt,
    output_file="self_consistency_security.json",
    n_runs=5
)

Saved -> outputs\self_consistency_security.json


## Tree of Thoughts

In [1]:
from prompts import (
    tree_of_thought_usecase_prompt,
    tree_of_thought_architecture_prompt
)

In [4]:
usecase_result = run_case(
    case_name="tree_of_thought_usecase",
    prompt_messages=tree_of_thought_usecase_prompt,
    output_filename="tot_usecase.json"
)



Running: tree_of_thought_usecase
Saved -> outputs\tot_usecase.json


In [5]:
architecture_result = run_case(
    case_name="tree_of_thought_architecture",
    prompt_messages=tree_of_thought_architecture_prompt,
    output_filename="tot_architecture.json"
)


Running: tree_of_thought_architecture
Saved -> outputs\tot_architecture.json


## Rephrase and Respond

In [1]:
from prompts import rephrase_business_prompt, rephrase_tech_prompt

In [3]:
business_result = run_case(
    case_name="rephrase_business",
    prompt_messages=rephrase_business_prompt,
    output_filename="rephrase_business.json",
    temperature=0.3
)


Running: rephrase_business
Saved -> outputs\rephrase_business.json


In [4]:
tech_result = run_case(
    case_name="rephrase_tech",
    prompt_messages=rephrase_tech_prompt,
    output_filename="rephrase_tech.json",
    temperature=0.3
)


Running: rephrase_tech
Saved -> outputs\rephrase_tech.json
